# EauRouge-F1 — Qwen3-8B Unsloth Train
10656 QA · 239 career_total · T4 16GB


In [ ]:
%%capture
!pip install -q unsloth trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
from datasets import load_dataset
path="/content/qa.jsonl" # wget https://raw.githubusercontent.com/VineshF1/F1-Dataset/main/data/final/qa.jsonl -O /content/qa.jsonl
ds=load_dataset("json", data_files=path)["train"].train_test_split(test_size=0.1, seed=42) # 9590/1066 for 10656
def fmt(examples):
    return {"text": [tokenizer.apply_chat_template([{"role":"user","content":q},{"role":"assistant","content":a}], tokenize=False, add_generation_prompt=False) for q,a in zip(examples["question"], examples["answer"])]}
ds=ds.map(fmt, batched=True)
print(ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer=SFTTrainer(model, tokenizer, train_dataset=ds["train"], eval_dataset=ds["test"], dataset_text_field="text", max_seq_length=1024,
  args=SFTConfig(per_device_train_batch_size=4, gradient_accumulation_steps=2, warmup_ratio=0.03, num_train_epochs=1.5, learning_rate=2e-4, fp16=True, packing=True, logging_steps=10, eval_steps=200, save_steps=200, output_dir="outputs", optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine", seed=3407, report_to="none"))
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)
def ask(q):
    p=tokenizer.apply_chat_template([{"role":"user","content":q}], tokenize=False, add_generation_prompt=True)
    o=model.generate(**tokenizer(p, return_tensors="pt").to("cuda"), max_new_tokens=150, temperature=0.1, top_p=0.9, repetition_penalty=1.1, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(o[0], skip_special_tokens=True).split("assistant")[-1].strip())
ask("who is 7 time world champion drivers?")
ask("Who is the only 5-time World Champion?")
ask("How many drivers have won 5 titles?")
ask("Is verstappen 4 time champion")
ask("How many drivers have won 4 titles?")


In [ ]:
!huggingface-cli login --token hf_xxx
model.push_to_hub("Vinesh/EauRouge-F1-Qwen3-8B-v2", token="hf_xxx")
tokenizer.push_to_hub("Vinesh/EauRouge-F1-Qwen3-8B-v2", token="hf_xxx")